In [ ]:
import import_ipynb
import pandas as pd
import numpy as np
from Association_Rule_Mining import create_continous_dataset
from scipy.stats import t as t_dist
from math import sqrt
import os
import tempfile
import mlflow
import matplotlib.pyplot as plt


In [ ]:
def calc_ema_volatility(df, span=7):
    prev_day_start = df.close.index.searchsorted(df.close.index - pd.Timedelta(days=1))
    prev_day_start = prev_day_start[prev_day_start > 0]
    prev_day_start = pd.Series(df.close.index[prev_day_start - 1], index=df.close.index[df.close.shape[0] - prev_day_start.shape[0]:])
    daily_returns = df.close.loc[prev_day_start.index] / df.close.loc[prev_day_start.values].values - 1
    vol = daily_returns.ewm(span=span).std()

    rolling_mean = vol.rolling(window=span).mean()
    return rolling_mean.mean()

In [ ]:
def calc_max_drawdown(df):
    df['return'] = df.close.pct_change()
    df['cum'] = (1 + df['return']).cumprod()
    running_max = df['cum'].cummax()
    mdd = ((df['cum'] - running_max) / running_max).min()
    return mdd

In [ ]:
def compare_drawdowns_and_volatilities(starting_date, ending_date):
    continous_df_btc = create_continous_dataset('BTC-USD', starting_date=starting_date, ending_date=ending_date)
    print(f"BTC-USD Volatility: {calc_ema_volatility(continous_df_btc)}")
    print(f"BTC-USD Max-Drawdown: {calc_max_drawdown(continous_df_btc)}")
    continous_df_msft = create_continous_dataset('MSFT', starting_date=starting_date, ending_date=ending_date)
    print(f"MSFT Volatility: {calc_ema_volatility(continous_df_msft)}")
    print(f"MSFT Max-Drawdown: {calc_max_drawdown(continous_df_msft)}")
    continous_df_amzn = create_continous_dataset('AMZN', starting_date=starting_date, ending_date=ending_date)
    print(f"AMZN Volatility: {calc_ema_volatility(continous_df_amzn)}")
    print(f"AMZN Max-Drawdown: {calc_max_drawdown(continous_df_amzn)}")
# compare_drawdowns_and_volatilities()

In [ ]:
def evaluate_buy_and_hold(market_data):
    df = market_data.copy()
    df["Return"] = df["close"].pct_change()
    df = df.dropna()

    # Buy once, hold all the way
    cumulative_return = (1 + df["Return"]).prod() - 1

    # Win rate = fraction of days with positive daily return
    win_rate = (df["Return"] > 0).mean() * 100

    return cumulative_return * 100, win_rate

In [ ]:
def evaluate_sma_crossover(market_data, short_window=10, long_window=30):
    df = market_data.copy()
    df["SMA_Short"] = df["close"].rolling(window=short_window).mean()
    df["SMA_Long"] = df["close"].rolling(window=long_window).mean()
    df = df.dropna()

    # Position: 1 = long, -1 = short (you can also use 0 for flat if you prefer)
    df["Position"] = np.where(df["SMA_Short"] > df["SMA_Long"], 1, -1)

    # Daily returns
    df["Return"] = df["close"].pct_change()
    df["Strategy_Return"] = df["Position"].shift(1) * df["Return"]

    # Metrics
    cumulative_return = (1 + df["Strategy_Return"]).prod() - 1
    win_rate = (df["Strategy_Return"] > 0).mean() * 100

    return cumulative_return * 100, win_rate

In [ ]:
def optimize_crossover(dataset):
    shorts = list(range(5,20))
    longs = list(range(20,50))

    best = 0
    short_wind = 0
    long_wind = 0

    for s in shorts:
        for l in longs:
            win_rate = evaluate_sma_crossover(pd.DataFrame(dataset, columns=["close"]), s, l)[1]
            if win_rate > best:
                best = win_rate
                short_wind = s
                long_wind = l
    return best, short_wind, long_wind
    
def get_baselines():
    df = pd.DataFrame()
    markets = ["AMZN", "BTC-USD", "MSFT"]
    short_range = ["2018-01-01", '2019-06-01']
    long_range = ["2018-01-01", '2021-01-01']
    for market in markets:
        short_df = create_continous_dataset(market, starting_date=short_range[0], ending_date=short_range[1])
        long_df = create_continous_dataset(market, starting_date=long_range[0], ending_date=long_range[1])
        short_cross = optimize_crossover(short_df)[0]
        short_wr = evaluate_buy_and_hold(short_df)[1]
        long_cross = optimize_crossover(long_df)[0]
        long_wr = evaluate_buy_and_hold(long_df)[1]
        df = pd.concat([df, pd.DataFrame({"Market": [market], "Name": f"{market}_BuyAndHold_Short", "Win-rate": short_wr})], ignore_index=True)
        df = pd.concat([df, pd.DataFrame({"Market": [market], "Name": f"{market}_BuyAndHold", "Win-rate": long_wr})], ignore_index=True)
        df = pd.concat([df, pd.DataFrame({"Market": [market], "Name": f"{market}_Crossover_Short", "Win-rate": short_cross})], ignore_index=True)
        df = pd.concat([df, pd.DataFrame({"Market": [market], "Name": f"{market}_Crossover", "Win-rate": long_cross})], ignore_index=True)
    return df


# btc_df = create_continous_dataset('BTC-USD', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
# btc_cross = optimize_crossover(btc_df)
# btc_wr = evaluate_buy_and_hold(btc_df)[1]
# print(f"BTC Buy and Hold Win Rate: {btc_cross / btc_wr * 100}%")

# amzn_df = create_continous_dataset('AMZN', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
# amzn_cross = optimize_crossover(amzn_df)
# amzn_wr = evaluate_buy_and_hold(amzn_df)[1]
# print(f"AMZN Buy and Hold Win Rate: {amzn_cross / amzn_wr * 100}%")


# msft_df = create_continous_dataset('MSFT', starting_date=STARTING_DATE, ending_date=ENDING_DATE)
# msft_cross = optimize_crossover(msft_df)
# msft_wr = evaluate_buy_and_hold(msft_df)[1]
# print(f"MSFT Buy and Hold Win Rate: {msft_cross / msft_wr * 100}%")

In [ ]:
def welch_t_test(mean1, std1, n1, mean2, std2, n2):
    # Welch t-statistic
    numerator = mean1 - mean2
    denominator = sqrt((std1**2)/n1 + (std2**2)/n2)
    
    if denominator == 0:
        return np.nan, np.nan
    
    t_stat = numerator / denominator

    # Degrees of freedom
    df_num = ((std1**2)/n1 + (std2**2)/n2)**2
    df_den = ((std1**2)**2) / (n1**2 * (n1-1)) + ((std2**2)**2) / (n2**2 * (n2-1))
    df = df_num / df_den

    # Two-tailed p-value
    p_value = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    return t_stat, p_value

In [ ]:
# #2018-2019
# #AMZN
# print("***AMZN***")
# print(welch_t_test(63.64, 3.21, 5, 66.21, 6.22, 5)) #Sentiment Only
# print(welch_t_test(64.55, 2.03, 5, 66.21, 6.22, 5)) #Technical Only
# print(welch_t_test(62.42, 3.28, 5, 66.21, 6.22, 5)) #TP High lift Only
# print(welch_t_test(65.81, 1.33, 5, 66.21, 6.22, 5)) #TP Low lift Only
# print(welch_t_test(57.45, 6.46, 5, 66.21, 6.22, 5)) #TP Zero lift Only
# #BTC-USD
# print("***BTC-USD***")
# print(welch_t_test(54.55, 0, 5, 66.15, 0.60, 5)) #Sentiment Only
# print(welch_t_test(55.35, 2.57, 5, 66.15, 0.60, 5)) #Technical Only
# print(welch_t_test(57.22, 4.66, 5, 66.15, 0.60, 5)) #TP High lift Only
# print(welch_t_test(51.14, 3.42, 5, 66.15, 0.60, 5)) #TP Low lift Only
# print(welch_t_test(60.27, 3.92, 5, 66.15, 0.60, 5)) #All Parameters
# #MSFT
# print("***MSFT***")
# print(welch_t_test(70, 0, 5, 75, 7.14, 5)) #Sentiment Only
# print(welch_t_test(62.50, 15, 5, 75, 7.14, 5)) #Technical Only
# print(welch_t_test(71.43, 3.89, 5, 75, 7.14, 5)) #TP High lift Only
# print(welch_t_test(64.77, 2.27, 5, 75, 7.14, 5)) #TP Zero lift Only
# print(welch_t_test(72.62, 5.99, 5, 75, 7.14, 5)) #All Parameters

In [ ]:
# amzn_shorter_period_values = [
#     {
#         "Name": "AMZN_Sentiment_Only",
#         "Market": "AMZN",
#         "Result Mean": 63.64,
#         "Result Std":  3.21
#     },
#     {
#         "Name": "AMZN_Technical_Only",
#         "Market": "AMZN",
#         "Result Mean": 64.55,
#         "Result Std":  2.03
#     },
#     {
#         "Name": "AMZN_TargetProfitable_Zero_Lift",
#         "Market": "AMZN",
#         "Result Mean": 57.45,
#         "Result Std": 6.46
#     },
#     {
#         "Name": "AMZN_TargetProfitable_Low_Lift",
#         "Market": "AMZN",
#         "Result Mean": 65.81,
#         "Result Std":  1.33
#     },
#     {
#         "Name": "AMZN_TargetProfitable_High_Lift",
#         "Market": "AMZN",
#         "Result Mean": 62.42,
#         "Result Std":  3.28
#     },
#     {
#         "Name": "AZMN_All_Params",
#         "Market": "AMZN",
#         "Result Mean": 66.21,
#         "Result Std":  6.22
#     },
#     {
#         "Name": "AMZN_SMA-crossover",
#         "Market": "AMZN",
#         "Result Mean": 60.22,
#         "Result Std":  0.00
#     },
# ]

# btcusd_shorter_period_values = [
#     {
#         "Name": "BTC_Sentiment_Only",
#         "Market": "BTC-USD",
#         "Result Mean": 54.55,
#         "Result Std":  0.00
#     },
#     {
#         "Name": "BTC_Technical_Only",
#         "Market": "BTC-USD",
#         "Result Mean": 55.35,
#         "Result Std":  2.57
#     },
#     {
#         "Name": "BTC_TargetProfitable_Zero_Lift",
#         "Market": "BTC-USD",
#         "Result Mean": 66.15,
#         "Result Std": 0.60
#     },
#     {
#         "Name": "BTC_TargetProfitable_Low_Lift",
#         "Market": "BTC-USD",
#         "Result Mean": 51.14,
#         "Result Std":  3.42
#     },
#     {
#         "Name": "BTC_TargetProfitable_High_Lift",
#         "Market": "BTC-USD",
#         "Result Mean": 57.22,
#         "Result Std":  4.66
#     },
#     {
#         "Name": "BTC_All_Params",
#         "Market": "BTC-USD",
#         "Result Mean": 60.27,
#         "Result Std":  3.92
#     },
#     {
#         "Name": "BTC_SMA-crossover",
#         "Market": "BTC-USD",
#         "Result Mean": 51.35,
#         "Result Std":  0.00
#     },
# ]

# msft_shorter_period_values = [
#     {
#         "Name": "MSFT_Sentiment_Only",
#         "Market": "MSFT",
#         "Result Mean": 70.00,
#         "Result Std":  0.00
#     },
#     {
#         "Name": "MSFT_Technical_Only",
#         "Market": "MSFT",
#         "Result Mean": 62.50,
#         "Result Std":  15
#     },
#     {
#         "Name": "MSFT_TargetProfitable_Zero_Lift",
#         "Market": "MSFT",
#         "Result Mean": 64.77,
#         "Result Std": 2.27
#     },
#     {
#         "Name": "MSFT_TargetProfitable_Low_Lift",
#         "Market": "MSFT",
#         "Result Mean": 75,
#         "Result Std":  7.14
#     },
#     {
#         "Name": "MSFT_TargetProfitable_High_Lift",
#         "Market": "MSFT",
#         "Result Mean": 71.43,
#         "Result Std":  3.89
#     },
#     {
#         "Name": "MSFT_All_Params",
#         "Market": "MSFT",
#         "Result Mean": 72.62,
#         "Result Std":  5.99
#     },
#     {
#         "Name": "MSFT_SMA-crossover",
#         "Market": "MSFT",
#         "Result Mean": 55.69,
#         "Result Std":  0.00
#     },
# ]

In [ ]:
# #BTC-USD 2018-2019 Full Results
# test_and_rel_performance(pd.DataFrame(btcusd_shorter_period_values), 50.79365079365079).sort_values(by="Result Mean", ascending=False)
# #BTC-USD 2018-2021 Full Results
# print_full_results("BTC-USD", 53.25047801147228)
# #AMZN 2018-2019 Full Results
# test_and_rel_performance(pd.DataFrame(amzn_shorter_period_values), 53.73134328358209).sort_values(by="Result Mean", ascending=False)
# #AMZN 2018-2021 Full Results
# print_full_results("AMZN", 54.39093484419264)
# #MSFT 2018-2019 Full Results
# test_and_rel_performance(pd.DataFrame(msft_shorter_period_values), 54.72636815920397).sort_values(by="Result Mean", ascending=False)
# #MSFT 2018-2021 Full Results
# print_full_results("MSFT", 56.798866855524075)

In [ ]:

def load_mlflow_wr_and_sr(experiment_name):
    # experiments = [
    #     'MSFT_Sentiment_Only',
    #     'MSFT_Technical_Only_Short', 
    #     'MSFT_TargetProfitable_High_Lift',
    #     'MSFT_TargetProfitable_Low_Lift',
    #     'MSFT_TargetProfitable_Zero_Lift',
    #     'MSFT_All_Params'
    # ]

    # results = []

    # for exp in experiments:

    # Set the experiment name
    experiment = mlflow.get_experiment_by_name(experiment_name)
    # Search all runs in the experiment
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

    # Filter runs (optional: e.g., only successful runs or specific tags)
    # runs = runs[runs['status'] == 'FINISHED']

    # Calculate the average of a specific metric
    average_metric_wr = runs['metrics.Win-Rate'].mean()
    std_metric_wr = runs['metrics.Win-Rate'].std()

    average_metric_sh = runs['metrics.Sharpe-Ratio'].mean()
    std_metric_sh = runs['metrics.Sharpe-Ratio'].std()
    # results.append([exp, average_metric_wr, std_metric_wr])
    # print(f"Experiment: {exp}")
    # print(f"Average Win Rate Metric: {average_metric_wr}, {std_metric_wr}")
    # print(f"Relative Win Rate Metric: {(average_metric_wr / msft_wr) * 100}%")
    # print(f"Average Sharpe Metric: {average_metric_sh}, {std_metric_sh}")
    # print("--------------------------------------------------")
    return average_metric_wr, std_metric_wr, average_metric_sh, std_metric_sh

In [ ]:
def get_mlflow_data(type='train', test_id=1):
    experiment = mlflow.get_experiment_by_name('NormalTESTINGv3')
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

    with tempfile.TemporaryDirectory() as tmp_dir:
        sharpe = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_sharpe_iterations.csv",
            dst_path=tmp_dir
        )
        
        sortino = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_sortino_iterations.csv",
            dst_path=tmp_dir
        )
        
        calmar_ratio = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_calmar_ratio_iterations.csv",
            dst_path=tmp_dir
        )
        
        win_rate = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_wr_iterations.csv",
            dst_path=tmp_dir
        )
        
        max_drawdown = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_max_drawdown_iterations.csv",
            dst_path=tmp_dir
        )
        
        cumulative_return = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_cumulative_return_iterations.csv",
            dst_path=tmp_dir
        )
        
        transactions = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_tx_iterations.csv",
            dst_path=tmp_dir
        )
        
        holds = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_holds_iterations.csv",
            dst_path=tmp_dir
        )
                
        shorts = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_shorts_iterations.csv",
            dst_path=tmp_dir
        )
        
        longs = mlflow.artifacts.download_artifacts(
            run_id=runs["run_id"][test_id],
            artifact_path=f"{type}_longs_iterations.csv",
            dst_path=tmp_dir
        )
        
        # Read CSV
        sh = pd.read_csv(sharpe)
        so = pd.read_csv(sortino)
        ca = pd.read_csv(calmar_ratio)
        wr = pd.read_csv(win_rate)
        md = pd.read_csv(max_drawdown)
        cr = pd.read_csv(cumulative_return)
        tx = pd.read_csv(transactions)
        ho = pd.read_csv(holds)
        sr = pd.read_csv(shorts)
        lo = pd.read_csv(longs)
        
        data = pd.concat([sh['sharpe_ratio'], so['sortino_ratio'], ca['calmar_ratio'], wr['win_rate'], md['max_drawdown'], cr['cumulative_return'], tx['transactions'], ho['holds'], sr['shorts'], lo['longs']], axis=1)
        
        return data

In [ ]:
data_1 = get_mlflow_data(type='val', test_id=1)
data_2 = get_mlflow_data(type='val', test_id=2)
data_3 = get_mlflow_data(type='val', test_id=3)
data_4 = get_mlflow_data(type='val', test_id=0)

In [ ]:
train_2 = get_mlflow_data(type='train', test_id=2)
val_2 = get_mlflow_data(type='val', test_id=2)
test_2 = get_mlflow_data(type='test', test_id=2)

In [ ]:
train_1 = get_mlflow_data(type='train', test_id=1)
val_1 = get_mlflow_data(type='val', test_id=1)
test_1 = get_mlflow_data(type='test', test_id=1)

In [ ]:
train_1.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

train = train_1[["win_rate", "sharpe_ratio", "cumulative_return", "transactions"]].add_prefix("train_")
val = val_1[["win_rate", "sharpe_ratio", "cumulative_return", "transactions"]].add_prefix("val_")
test = test_1[["win_rate", "sharpe_ratio", "cumulative_return", "transactions"]].add_prefix("test_")

conc = pd.concat([train, val, test], axis=1)
conc = conc[
    (conc["val_transactions"] >= 10) &
    (conc["train_transactions"] >= 10)
]

conc = conc.nlargest(10, "val_sharpe_ratio")

conc[["train_sharpe_ratio", "val_sharpe_ratio", "test_sharpe_ratio"]].plot(kind="bar", ax=ax)

plt.show()

In [ ]:
conc.describe()

In [ ]:
train = train_1[["win_rate", "sharpe_ratio", "cumulative_return", "transactions", "shorts", "longs", "calmar_ratio"]].add_prefix("train_")
val = val_1[["win_rate", "sharpe_ratio", "cumulative_return", "transactions", "shorts", "longs", "calmar_ratio"]].add_prefix("val_")
test = test_1[["win_rate", "sharpe_ratio", "cumulative_return", "transactions", "shorts", "longs", "calmar_ratio"]].add_prefix("test_")

conc = pd.concat([train, val, test], axis=1)

conc = conc[conc["val_transactions"] >= 30]
conc["score"] = conc["val_sharpe_ratio"] + 0.5 * conc["val_calmar_ratio"]

# # top 10 po validation Sharpe
# conc = conc.nlargest(10, "val_sharpe_ratio")

# tabela wynikowa
conc[["val_sharpe_ratio", "val_calmar_ratio", "test_sharpe_ratio", "test_calmar_ratio", "test_transactions", "test_shorts", "test_longs", "score"]].sort_values(by="score", ascending=False)

In [ ]:
conc[["train_transactions", "val_transactions", "test_transactions"]].value_counts()

In [ ]:
conc[["train_sharpe_ratio", "val_sharpe_ratio", "test_sharpe_ratio"]].corr()

In [ ]:
conc[["train_win_rate", "val_win_rate", "test_win_rate"]].corr()

In [ ]:

fig, ax1 = plt.subplots(figsize=(10,6))

# plot
ax1.plot(train_2.index, train_2["sharpe_ratio"], color="green", marker="o", alpha=0.6)
ax1.set_ylabel("Training Sharpe Ratio")

ax3 = ax1.twinx()
ax3.plot(test_2.index, test_2["sharpe_ratio"], color="blue", marker="o")
ax3.set_ylabel("Test Sharpe Ratio")
plt.show()

In [ ]:
data_1.describe()

In [ ]:
data_2.describe()

In [ ]:
data_3.describe()

In [ ]:
summary_idx = pd.DataFrame({
    "min_index": train_2.idxmin(),
    "max_index": train_2.idxmax()
})
print(summary_idx)

In [ ]:
summary_idx = pd.DataFrame({
    "min_index": val_2.idxmin(),
    "max_index": val_2.idxmax()
})
print(summary_idx)

In [ ]:
summary_idx = pd.DataFrame({
    "min_index": test_2.idxmin(),
    "max_index": test_2.idxmax()
})
print(summary_idx)

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

# pierwsza oś
data_1[["sharpe_ratio", "win_rate"]].plot(ax=ax)

# druga oś
ax2 = ax.twinx()
data_1["transactions"].plot(ax=ax2, color="black")

plt.show()

In [ ]:
data_1.plot(y=["sharpe_ratio", "sortino_ratio", "calmar_ratio", "cumulative_return", "win_rate"], figsize=(10,10))

In [ ]:
data_1.plot(subplots=True, layout=(4,4), figsize=(10,10))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in data_1.select_dtypes(include='number').columns:
    sns.histplot(data_1[col], kde=True)
    plt.title(col)
    plt.show()

In [ ]:

fig, ax1 = plt.subplots(figsize=(10,6))

# barplot
ax1.bar(data_1.index, data_1["win_rate"], alpha=0.6)
ax1.set_ylabel("Win Rate")

# druga oś
ax2 = ax1.twinx()
ax2.plot(data_1.index, data_1["transactions"], color="red", marker="o")
ax2.set_ylabel("Transactions")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(data_1.corr(), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
train_2_pref = train_2[["win_rate", "sharpe_ratio", "transactions"]].add_prefix("train_")
test_2_pref = test_2[["win_rate", "sharpe_ratio", "transactions"]].add_prefix("test_")

sns.pairplot(pd.concat([train_2_pref, test_2_pref], axis=1), height=1.5)
plt.show()

In [ ]:
def load_mlflow_cum_returns(experiment):
    # experiments = [
    #     'MSFT_Sentiment_Only_Short',
    #     'MSFT_Technical_Only_Short', 
    #     'MSFT_TargetProfitable_High_Lift_Short',
    #     'MSFT_TargetProfitable_Low_Lift_Short',
    #     'MSFT_TargetProfitable_Zero_Lift_Short',
    #     'MSFT_All_Params_Short'
    # ]

    # for exp in experiments:
    experiment = mlflow.get_experiment_by_name(experiment)

    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

    values = []

    for _, run in runs.iterrows():
        run_id = run["run_id"]
        
        try:
            # Download artifact to temp directory
            with tempfile.TemporaryDirectory() as tmp_dir:
                local_path = mlflow.artifacts.download_artifacts(
                    run_id=run_id,
                    artifact_path="cum_returns.csv",
                    dst_path=tmp_dir
                )
                
                # Read CSV
                df = pd.read_csv(local_path)
                
                # Get last value from cumulative_return column
                last_value = df["cumulative_return"].iloc[-1]
                values.append(last_value)
                
        except Exception as e:
            print(f"Skipping run {run_id}: {e}")

    # Convert to pandas Series
    values = pd.Series(values)

    mean_value = values.mean()
    std_value = values.std()
    
    return mean_value, std_value

In [ ]:
def build_results():
    cases = pd.read_csv("../TestCases/DDQN_Test_Cases.csv")[2:6]
    mapped_df = pd.DataFrame()
    pd.options.display.float_format = "{:.3f}".format
    cases = cases[~cases["TestCaseName"].str.contains("LASSO", na=False) & ~cases["TestCaseName"].str.contains("PCA", na=False)]
    cases["TestCaseName"] = cases["TestCaseName"] + "_return_classification_v2"
    # cases["TestCaseName"] = cases["TestCaseName"].str.split("_Short").str[0]
    mapped_df["Name"] = cases["TestCaseName"]
    mapped_df["Market"] = cases["Market"]
    mapped_df["Win-rate"] = cases["TestCaseName"].apply(lambda x: float(f"{load_mlflow_wr_and_sr(x)[0]:.2f}"))
    mapped_df["Win-rate-std"] = cases["TestCaseName"].apply(lambda x: float(f"{load_mlflow_wr_and_sr(x)[1]:.2f}"))
    idx = mapped_df.groupby("Market")["Win-rate"].idxmax()
    max_rows = mapped_df.loc[idx, ["Market", "Win-rate", "Win-rate-std"]]
    mapped_df["t-statistic"] = mapped_df.apply(lambda x: welch_t_test(x["Win-rate"], x["Win-rate-std"], 5, max_rows.loc[max_rows["Market"] == x["Market"], "Win-rate"].iloc[0], max_rows.loc[max_rows["Market"] == x["Market"], "Win-rate-std"].iloc[0], 5)[0], axis=1)
    mapped_df["p-value"] = mapped_df.apply(lambda x: welch_t_test(x["Win-rate"], x["Win-rate-std"], 5, max_rows.loc[max_rows["Market"] == x["Market"], "Win-rate"].iloc[0], max_rows.loc[max_rows["Market"] == x["Market"], "Win-rate-std"].iloc[0], 5)[1], axis=1)
    mapped_df["Sharpe-Ratio"] = cases["TestCaseName"].apply(lambda x: float(f"{load_mlflow_wr_and_sr(x)[2]:.3f}"))
    mapped_df["Sharpe-Ratio-std"] = cases["TestCaseName"].apply(lambda x: float(f"{load_mlflow_wr_and_sr(x)[3]:.3f}"))
    mapped_df["Cumulative-Return"] = cases["TestCaseName"].apply(lambda x: float(f"{load_mlflow_cum_returns(x)[0]:.3f}"))
    mapped_df["Cumulative-Return-std"] = cases["TestCaseName"].apply(lambda x: float(f"{load_mlflow_cum_returns(x)[1]:.3f}"))
    return mapped_df

In [ ]:
vals = build_results()

In [ ]:
vals

In [ ]:
def build_relative_wr_heatmap():
    cases = pd.read_csv("../TestCases/DDQN_Test_Cases.csv")
    mapped_df = pd.DataFrame()
    pd.options.display.float_format = "{:.3f}".format
    cases = cases[~cases["TestCaseName"].str.contains("LASSO", na=False) & ~cases["TestCaseName"].str.contains("PCA", na=False)]
    mapped_df["Name"] = pd.concat([cases["TestCaseName"], cases["TestCaseName"].str.split("_Short").str[0]], ignore_index=True)
    mapped_df["Market"] = pd.concat([cases["Market"], cases["Market"]], ignore_index=True)
    mapped_df["Win-rate"] = mapped_df["Name"].apply(lambda x: float(f"{load_mlflow_wr_and_sr(x)[0]:.2f}"))
    baselines = get_baselines()
    mapped_df = pd.concat([mapped_df, baselines], ignore_index=True)
    mapped_df["Relative-Win-rate"] = mapped_df.apply(
        lambda row: row['Win-rate'] / mapped_df[mapped_df["Name"] == f"{row['Market']}_BuyAndHold_Short"]["Win-rate"].iloc[0] * 100 if   row["Name"].split("_")[-1] == "Short" else row['Win-rate'] / mapped_df[mapped_df["Name"] == f"{row['Market']}_BuyAndHold"]["Win-rate"].iloc[0] * 100, axis=1)


    df = mapped_df.copy()

    # --- REMOVE BUY & HOLD ---
    df = df[~df["Name"].str.contains("BuyAndHold")].copy()

    # --- EXTRACT DIRECTION ---
    df["Direction"] = np.where(df["Name"].str.contains("_Short"), "short", "long")

    # --- CLEAN NAME ---
    df["BaseName"] = df["Name"].str.replace("_Short", "", regex=False)
    df["Strategy"] = df["BaseName"].str.split("_", 1).str[1]

    # --- CREATE COLUMN LABELS ---
    df["Column"] = df["Market"] + " " + df["Direction"]
    df["Column"] = df["Column"].str.replace(" long", "", regex=False)

    # --- PIVOT ---
    heatmap_data = df.pivot_table(
        index="Strategy",
        columns="Column",
        values="Relative-Win-rate"
    )

    # --- ORDER ROWS ---
    row_order = [
        "Crossover",
        "All_Params",
        "TargetProfitable_Zero_Lift",
        "TargetProfitable_Low_Lift",
        "TargetProfitable_High_Lift",
        "Technical_Only",
        "Sentiment_Only",
    ]

    heatmap_data = heatmap_data.reindex(row_order)

    # --- ORDER COLUMNS ---
    col_order = [
        "AMZN short", "AMZN",
        "BTC-USD short", "BTC-USD",
        "MSFT short", "MSFT"
    ]

    heatmap_data = heatmap_data[col_order]

    # Optional: rename BTC-USD → BTC
    heatmap_data.columns = [c.replace("BTC-USD", "BTC") for c in heatmap_data.columns]


    # =========================
    #        HEATMAP
    # =========================

    plt.figure(figsize=(11, 6))

    vmin = heatmap_data.min().min()
    vmax = heatmap_data.max().max()

    im = plt.imshow(heatmap_data, aspect="auto", vmin=vmin-10, vmax=vmax+10)

    plt.colorbar(im, label="Relative Win Rate")

    plt.xticks(range(len(heatmap_data.columns)), heatmap_data.columns, rotation=45)
    plt.yticks(range(len(heatmap_data.index)), heatmap_data.index)

    plt.title("Relative Win Rate Heatmap")

    # -------- INSERT VALUES INSIDE CELLS --------
    for i in range(len(heatmap_data.index)):
        for j in range(len(heatmap_data.columns)):
            value = heatmap_data.iloc[i, j]
            plt.text(
                j, i,
                f"{value:.2f}",
                ha="center",
                va="center",
                color="white"
            )

    plt.tight_layout()
    plt.show()

In [ ]:
build_relative_wr_heatmap()